In [4]:
import random
import heapq

In [5]:
def get_conflicts(state):
    conflicts = 0
    n = len(state)
    for i in range(n):
        for j in range(i + 1, n):
            if state[i] == state[j] or abs(state[i] - state[j]) == abs(i - j):
                conflicts += 1
    return conflicts

def hill_climbing(n):
    current_state = [random.randint(0, n-1) for _ in range(n)]
    current_conflicts = get_conflicts(current_state)
    
    while True:
        neighbor = list(current_state)
        best_neighbor = list(current_state)
        min_conflicts = current_conflicts
        
        for col in range(n):
            for row in range(n):
                if current_state[col] == row: continue
                neighbor[col] = row
                new_conflicts = get_conflicts(neighbor)
                if new_conflicts < min_conflicts:
                    min_conflicts = new_conflicts
                    best_neighbor = list(neighbor)
        
        if min_conflicts >= current_conflicts:
            break
            
        current_state = best_neighbor
        current_conflicts = min_conflicts
        
    return current_state, current_conflicts

solution, score = hill_climbing(8)
print(f"Final State: {solution} with {score} conflicts.")

Final State: [0, 2, 6, 6, 1, 1, 3, 1] with 7 conflicts.


In [6]:

num_staff = 10
num_shifts = 7
population_size = 50
mutation_rate = 0.1
max_generations = 100

def create_random_schedule():
    return [[random.randint(0, 1) for _ in range(num_shifts)] for _ in range(num_staff)]

def evaluate_fitness(schedule):
    conflicts = 0
    for j in range(num_shifts):
        shift_coverage = sum(schedule[i][j] for i in range(num_staff))
        if shift_coverage == 0:
            conflicts += 1
    for i in range(num_staff):
        if sum(schedule[i]) > 2:
            conflicts += 1
            
    return conflicts

def select_parents(population, fitness_scores):
    sorted_population = [x for _, x in sorted(zip(fitness_scores, population))]
    return sorted_population[:population_size // 2]

def crossover(parent1, parent2):
    cp = random.randint(1, num_staff - 1)
    child = parent1[:cp] + parent2[cp:]
    return child

def mutate(schedule):
    staff_idx = random.randint(0, num_staff - 1)
    s1, s2 = random.sample(range(num_shifts), 2)
    schedule[staff_idx][s1], schedule[staff_idx][s2] = schedule[staff_idx][s2], schedule[staff_idx][s1]
    return schedule

population = [create_random_schedule() for _ in range(population_size)]

for generation in range(max_generations):
    fitness_scores = [evaluate_fitness(ind) for ind in population]
    best_fitness = min(fitness_scores)
    
    if best_fitness == 0:
        print(f"Optimal solution found in generation {generation}!")
        break
        
    parents = select_parents(population, fitness_scores)
    new_population = []
    while len(new_population) < population_size:
        p1, p2 = random.sample(parents, 2)
        child = crossover(p1, p2)
        if random.random() < mutation_rate:
            child = mutate(child)
        new_population.append(child)
    population = new_population

best_schedule = population[fitness_scores.index(min(fitness_scores))]
print("Best Schedule: ", best_schedule)

Optimal solution found in generation 7!
Best Schedule:  [[0, 0, 0, 0, 1, 0, 0], [1, 0, 0, 1, 0, 0, 0], [0, 0, 1, 0, 0, 0, 1], [1, 1, 0, 0, 0, 0, 0], [0, 0, 0, 1, 0, 0, 0], [0, 0, 0, 0, 0, 1, 1], [0, 0, 0, 0, 0, 1, 1], [0, 0, 0, 1, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0], [1, 1, 0, 0, 0, 0, 0]]


In [8]:
graph = {
    'S': [('A', 3), ('B', 6), ('C', 5)],
    'A': [('D', 9), ('E', 8)],
    'B': [('F', 12), ('G', 14)],
    'C': [('H', 7)],
    'H': [('I', 5), ('J', 6)],
    'I': [('K', 1), ('L', 10), ('M', 2)],
    'D': [], 'E': [], 'F': [], 'G': [], 
    'J': [], 'K': [], 'L': [], 'M': []
}

def beam_search(start, goal, beam_width=2):
    beam = [(0, [start])]

    while beam:
        candidates = []
        for cost, path in beam:
            current_node = path[-1]
            
            if current_node == goal:
                return path, cost
            for neighbor, edge_cost in graph.get(current_node, []):
                new_cost = cost + edge_cost
                new_path = path + [neighbor]
                candidates.append((new_cost, new_path))
        
        if not candidates:
            break
        beam = heapq.nsmallest(beam_width, candidates, key=lambda x: x[0])
    
    return None, float('inf')

path, total_cost = beam_search(start='S', goal='L', beam_width=3)

if path:
    print(f"Path found: {' -> '.join(path)} with total cost: {total_cost}")
else:
    print("No path found.")

Path found: S -> C -> H -> I -> L with total cost: 27
